# Pricing Data Processing Pipeline
## Bronze → Silver → Gold → Master Dimension

### 🚀 Ready to Run!

**1. S3 Access:**
   - ✅ Unity Catalog credentials configured!
   - Credential: `db_s3_credentials_databricks-s3-ingest-9db34`
   - No manual setup needed

**2. Data Source:**
   - S3 Bucket: `sportx-bar` (region: ap-southeast-2)
   - Path: `s3://sportx-bar/gross_price/*.csv`
   - All CSV files in the folder will be loaded and concatenated

**3. Required files:**
   - Products dimension: `data/silver_products.csv` (from products notebook)

**4. Output files:**
   All processed data will be saved to the `data/` subfolder:
   - `data/bronze_gross_price.csv` - Raw data with metadata
   - `data/silver_gross_price.csv` - Cleaned and transformed data
   - `data/gold_sb_dim_gross_price.csv` - Business dimension table
   - `data/gold_dim_gross_price.csv` - Master dimension table

**5. Run cells sequentially** from top to bottom.

---

In [0]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("✓ Core libraries imported")

In [0]:
# Define parameters
catalog = "fmcg"
data_source = "gross_price"

# S3 bucket with Unity Catalog credentials
s3_bucket = "sportx-bar"  # Using Unity Catalog credential: db_s3_credentials_databricks-s3-ingest-9db34

print(f"Catalog: {catalog}")
print(f"Data source: {data_source}")
print(f"S3 path: s3://{s3_bucket}/{data_source}/*.csv")

# Create data directory for outputs
os.makedirs("data", exist_ok=True)

## Bronze

In [0]:
# Read CSV from S3 using Spark (automatically uses Unity Catalog credentials)
# Then convert to pandas for processing

# Build S3 path
s3_path = f's3://{s3_bucket}/{data_source}/*.csv'
print(f"Reading from S3: {s3_path}")
print(f"Using Unity Catalog credential: db_s3_credentials_databricks-s3-ingest-9db34")

try:
    # Read CSV from S3 using Spark (UC credentials work automatically)
    df_spark = spark.read.csv(s3_path, header=True, inferSchema=True)
    
    # Convert to pandas DataFrame
    df = df_spark.toPandas()
    
    print(f"\n✓ Successfully loaded {len(df)} rows")
    
    # Add Bronze metadata columns
    df["read_timestamp"] = pd.Timestamp.now()
    df["file_name"] = f"{data_source}.csv"
    df["file_size"] = None
    
    print(f"\nColumns: {list(df.columns)}")
    print(f"Shape: {df.shape}")
    
except Exception as e:
    print(f"\n✗ Error reading from S3: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Verify file exists at: {s3_path}")
    print(f"  2. Check Unity Catalog external location is configured for: s3://{s3_bucket}/")
    print(f"  3. Ensure credential has permissions: s3:GetObject, s3:ListBucket")
    raise

In [0]:
# Display data types
print("\nData types:")
print(df.dtypes)
print(f"\nShape: {df.shape}")

In [0]:
# Display first 10 rows
display(df.head(10))

In [0]:
# Save Bronze layer to CSV
output_path = f"data/bronze_{data_source}.csv"
df.to_csv(output_path, index=False)

print(f"✓ Bronze layer saved to: {output_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

## Silver

In [0]:
# Load Bronze data from CSV
df_bronze = pd.read_csv(f"data/bronze_{data_source}.csv")
print(f"Loaded Bronze data: {len(df_bronze)} rows")
print(f"\nFirst 10 rows:")
display(df_bronze.head(10))

**Transformations**

- 1: Normalise `month` field

In [0]:
# Check unique month values before transformation
print("Unique month values:")
print(df_bronze['month'].unique())

In [0]:
# 1️⃣ Parse `month` from multiple possible formats
df_silver = df_bronze.copy()

# Try parsing with different date formats
date_formats = ['%Y/%m/%d', '%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y']

def parse_date(date_str):
    """Try to parse date with multiple formats"""
    if pd.isna(date_str):
        return None
    
    for fmt in date_formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    return None

# Apply the parsing function
df_silver['month'] = df_silver['month'].apply(parse_date)

print("✓ Normalized month field")
print(f"  Successfully parsed: {df_silver['month'].notna().sum()} rows")
print(f"  Failed to parse: {df_silver['month'].isna().sum()} rows")

In [0]:
# Check unique month values after transformation
print("\nUnique month values (first 20):")
print(df_silver['month'].unique()[:20])

- 2: Handling `gross_price`

In [0]:
# Display first 10 rows
display(df_silver.head(10))

In [0]:
# 2️⃣ Clean and validate gross_price column
# - Convert valid numeric values to float
# - Fix negative prices by making them positive
# - Replace non-numeric values with 0

import re

def clean_price(price):
    """Clean and validate price value"""
    if pd.isna(price):
        return 0.0
    
    # Convert to string for regex check
    price_str = str(price)
    
    # Check if it matches numeric pattern (including negative)
    if re.match(r'^-?\d+(\.\d+)?$', price_str):
        price_float = float(price_str)
        # Fix negative prices
        return abs(price_float)
    else:
        return 0.0

df_silver['gross_price'] = df_silver['gross_price'].apply(clean_price)

print("✓ Cleaned gross_price column")
print(f"  Valid prices: {(df_silver['gross_price'] > 0).sum()}")
print(f"  Zero prices: {(df_silver['gross_price'] == 0).sum()}")

In [0]:
# Display first 10 rows after cleaning
display(df_silver.head(10))

In [0]:
# 3️⃣ Enrich with product_code from products dimension
# Perform inner join to get product_code for each product_id

# Load products dimension
products_path = "data/silver_products.csv"

if not os.path.exists(products_path):
    raise FileNotFoundError(
        f"\n{'='*60}\n"
        f"Products dimension file not found: {products_path}\n\n"
        f"Please run the products notebook first to generate:\n"
        f"  data/silver_products.csv\n"
        f"{'='*60}\n"
    )

df_products = pd.read_csv(products_path)
print(f"Loaded products dimension: {len(df_products)} products")

# Join with products to get product_code
df_joined = df_silver.merge(
    df_products[['product_id', 'product_code']], 
    on='product_id', 
    how='inner'
)

# Select and reorder columns
df_joined = df_joined[['product_id', 'product_code', 'month', 'gross_price', 'read_timestamp', 'file_name', 'file_size']]

print(f"\n✓ Joined with products dimension")
print(f"  Rows before join: {len(df_silver)}")
print(f"  Rows after join: {len(df_joined)}")
print(f"  Matched products: {df_joined['product_id'].nunique()}")

display(df_joined.head(5))

In [0]:
# Save Silver layer to CSV
output_path = f"data/silver_{data_source}.csv"
df_joined.to_csv(output_path, index=False)

print(f"✓ Silver layer saved to: {output_path}")
print(f"  Rows: {len(df_joined)}")
print(f"  Columns: {len(df_joined.columns)}")

## Gold

In [0]:
# Load Silver data from CSV
df_silver = pd.read_csv(f"data/silver_{data_source}.csv")
print(f"Loaded Silver data: {len(df_silver)} rows")

In [0]:
# Select only required columns for Gold layer
df_gold = df_silver[['product_code', 'month', 'gross_price']].copy()

print(f"\nGold layer preview:")
display(df_gold.head(5))

In [0]:
# Save Gold layer to CSV
output_path = f"data/gold_sb_dim_{data_source}.csv"
df_gold.to_csv(output_path, index=False)

print(f"✓ Gold layer saved to: {output_path}")
print(f"  Rows: {len(df_gold)}")
print(f"  Columns: {len(df_gold.columns)}")

## Merging Data source with parent

In [0]:
# Load Gold data for master dimension processing
df_gold_price = pd.read_csv(f"data/gold_sb_dim_{data_source}.csv")
print(f"Gold price data loaded: {len(df_gold_price)} rows")
print(f"\nGold price preview:")
display(df_gold_price.head(5))

- Get the price for each product_code (aggregated by year)

In [0]:
# Calculate latest non-zero price for each product per year
# Priority: non-zero prices > most recent month

# Convert month to datetime if it's not already
df_gold_price['month'] = pd.to_datetime(df_gold_price['month'])

# Extract year from month
df_gold_price['year'] = df_gold_price['month'].dt.year

# Add is_zero flag: 0 = non-zero price (preferred), 1 = zero price
df_gold_price['is_zero'] = (df_gold_price['gross_price'] == 0).astype(int)

# Sort by: product_code, year, is_zero (non-zero first), month (most recent first)
df_gold_price_sorted = df_gold_price.sort_values(
    by=['product_code', 'year', 'is_zero', 'month'],
    ascending=[True, True, True, False]  # is_zero=True means 0 comes before 1 (non-zero before zero)
)

# Group by product_code and year, take the first row (highest priority)
df_gold_latest_price = df_gold_price_sorted.groupby(['product_code', 'year']).first().reset_index()

print(f"✓ Calculated latest prices")
print(f"  Total price records: {len(df_gold_price)}")
print(f"  Unique product-year combinations: {len(df_gold_latest_price)}")
print(f"  Products with prices: {df_gold_latest_price['product_code'].nunique()}")

In [0]:
# Display latest prices
print(f"\nLatest prices preview:")
display(df_gold_latest_price.head(10))

In [0]:
# Select required columns and rename
df_gold_latest_price = df_gold_latest_price[['product_code', 'year', 'gross_price']].copy()
df_gold_latest_price = df_gold_latest_price.rename(columns={'gross_price': 'price_inr'})

# Reorder columns
df_gold_latest_price = df_gold_latest_price[['product_code', 'price_inr', 'year']]

# Convert year to string
df_gold_latest_price['year'] = df_gold_latest_price['year'].astype(str)

print(f"\nFinal master dimension shape: {df_gold_latest_price.shape}")
print(f"Columns: {list(df_gold_latest_price.columns)}")
display(df_gold_latest_price.head(5))

In [0]:
# Display data types
print("\nData types:")
print(df_gold_latest_price.dtypes)

In [0]:
# Merge with Master dimension table (upsert logic)
# - Update existing product_code rows
# - Insert new product_code rows

master_path = f"data/gold_dim_{data_source}.csv"

# Load existing master table if it exists
if os.path.exists(master_path):
    df_master = pd.read_csv(master_path)
    print(f"Existing master table loaded: {len(df_master)} rows")
    master_rows_before = len(df_master)
    
    # Merge: Update existing, add new
    # Remove old rows for products that are being updated
    df_master = df_master[~df_master['product_code'].isin(df_gold_latest_price['product_code'])]
    
    # Append new/updated rows
    df_merged = pd.concat([df_master, df_gold_latest_price], ignore_index=True)
else:
    print("No existing master table found. Creating new one.")
    df_merged = df_gold_latest_price.copy()
    master_rows_before = 0

# Save merged master table
df_merged.to_csv(master_path, index=False)

print(f"\n✓ Master dimension table updated: {master_path}")
print(f"  Total rows: {len(df_merged)}")
print(f"  Child rows processed: {len(df_gold_latest_price)}")
print(f"  Master rows before: {master_rows_before}")
print(f"  Master rows after: {len(df_merged)}")

print(f"\n{'='*60}")
print("PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"Bronze: data/bronze_{data_source}.csv")
print(f"Silver: data/silver_{data_source}.csv")
print(f"Gold:   data/gold_sb_dim_{data_source}.csv")
print(f"Master: data/gold_dim_{data_source}.csv")